In [1]:
import os
import pandas as pd
import numpy as np
import arcpy
arcpy.env.overwriteOutput = True

In [2]:
nhd_gdb = r"C:/Users/Chels/OneDrive - University of Illinois - Urbana/Ch5_Sackett_Impacts/Wetland Mapping/Illinois_NHD_Shapefiles.gdb"
project_folder = r"C:/Users/Chels/OneDrive - University of Illinois - Urbana/Ch5_Sackett_Impacts/JEMA-Paper-Results/NHD_Data"

# Use iterative rules to assign a flow permanence category to unknown flowlines

## NHD flowline attribute table with 0's unassigned

In [ ]:
nhd_table_path = os.path.normpath(os.path.join(project_folder, "NHDFlowline_IL_Step4_HydroClass.csv"))

In [ ]:
nhd_table = pd.read_csv(nhd_table_path)

In [ ]:
# count starting number of flowlines in each class
start_zero_count = sum(nhd_table['NHD_Hydro_Class'] == 0)
start_perennial_count = sum(nhd_table['NHD_Hydro_Class'] == 1)
start_intermittent_count = sum(nhd_table['NHD_Hydro_Class'] == 2)
start_ephemeral_count = sum(nhd_table['NHD_Hydro_Class'] == 3)

In [ ]:
# number of rows in flowline dataframe
n_row = len(nhd_table)

## Assign downstream hydrologic class

In [ ]:
# create empty column
nhd_table['Downstream_Hydro_Class'] = 0

In [ ]:
# assign based on NHD major and minor downstream class
for i in range(n_row):
  dn_major_seq_i = nhd_table.loc[i,'dnhydroseq'].item()
  dn_minor_seq_i = nhd_table.loc[i,'dnminorhyd'].item()
  
  dn_major_class_i = nhd_table.loc[nhd_table['hydroseq'] == dn_major_seq_i,'NHD_Hydro_Class']
  dn_minor_class_i = nhd_table.loc[nhd_table['hydroseq'] == dn_minor_seq_i,'NHD_Hydro_Class']

  if dn_major_seq_i != 0:
    if (len(dn_major_class_i) > 0):
      nhd_table.loc[i,'Downstream_Hydro_Class'] = int(dn_major_class_i.iloc[0])
  elif dn_minor_seq_i != 0:
    if (len(dn_minor_class_i) > 0):
      nhd_table.loc[i,'Downstream_Hydro_Class'] = int(dn_minor_class_i.iloc[0])

In [ ]:
sum(nhd_table['Downstream_Hydro_Class'] == 1)

In [ ]:
sum(nhd_table['Downstream_Hydro_Class'] == 2)

In [ ]:
sum(nhd_table['Downstream_Hydro_Class'] == 3)

In [ ]:
sum(nhd_table['Downstream_Hydro_Class'] == 0)

## Assign upstream hydrologic class

In [ ]:
# create empty column
nhd_table['Upstream_Hydro_Class'] = 0

In [ ]:
for i in range(n_row):
  up_seq_i = nhd_table.loc[i,'uphydroseq'].item()
  up_class_i = nhd_table.loc[nhd_table['hydroseq'] == up_seq_i,'NHD_Hydro_Class']
  if up_seq_i != 0:
    if (len(up_class_i) > 0):
      nhd_table.loc[i,'Upstream_Hydro_Class'] = int(up_class_i.iloc[0])

In [ ]:
sum(nhd_table['Upstream_Hydro_Class'] == 1)

In [ ]:
sum(nhd_table['Upstream_Hydro_Class'] == 2)

In [ ]:
sum(nhd_table['Upstream_Hydro_Class'] == 3)

In [ ]:
sum(nhd_table['Upstream_Hydro_Class'] == 0)

## Assign final hydrologic class

In [ ]:
nhd_table['Final_NHD_Hydro_Class'] = 0

In [ ]:
for i in range(n_row):
  hydro_class_i = nhd_table.loc[i,"NHD_Hydro_Class"].item()
  up_class_i = nhd_table.loc[i,'Upstream_Hydro_Class'].item()
  down_class_i = nhd_table.loc[i,'Downstream_Hydro_Class'].item()
  if (hydro_class_i != 0):
    nhd_table.loc[i,'Final_NHD_Hydro_Class'] = hydro_class_i
  elif ((down_class_i != 0) & (up_class_i != 0)):
    if (down_class_i == up_class_i):
      nhd_table.loc[i,'Final_NHD_Hydro_Class'] = down_class_i
    elif (down_class_i < up_class_i):
      nhd_table.loc[i,'Final_NHD_Hydro_Class'] = up_class_i
    elif (down_class_i > up_class_i):
      nhd_table.loc[i,'Final_NHD_Hydro_Class'] = down_class_i
  elif (down_class_i != 0):
    if (down_class_i < 3):
      nhd_table.loc[i,'Final_NHD_Hydro_Class'] = down_class_i + 1
    elif (down_class_i == 3):
      nhd_table.loc[i,'Final_NHD_Hydro_Class'] = down_class_i
  elif (up_class_i != 0):
    nhd_table.loc[i,'Final_NHD_Hydro_Class'] = up_class_i

In [ ]:
sum(nhd_table['Final_NHD_Hydro_Class'] == 1)

In [ ]:
sum(nhd_table['Final_NHD_Hydro_Class'] == 2)

In [ ]:
sum(nhd_table['Final_NHD_Hydro_Class'] == 3)

In [ ]:
sum(nhd_table['Final_NHD_Hydro_Class'] == 4)

## Repeat until convergence

In [ ]:
# vector to keeep track of convergence
hydro_class_zeros = [sum(nhd_table['NHD_Hydro_Class'] == 0), sum(nhd_table['Final_NHD_Hydro_Class'] == 0)]
hydro_class_zeros

In [ ]:
for k in range(30):
    # assign downstream hydrologic class
    downstream_zero_rows = np.where(nhd_table['Downstream_Hydro_Class'] == 0)[0]
    for i in downstream_zero_rows:
      dn_major_seq_i = nhd_table.loc[i,'dnhydroseq'].item()
      dn_minor_seq_i = nhd_table.loc[i,'dnminorhyd'].item()
      
      dn_major_class_i = nhd_table.loc[nhd_table['hydroseq'] == dn_major_seq_i,'Final_NHD_Hydro_Class']
      dn_minor_class_i = nhd_table.loc[nhd_table['hydroseq'] == dn_minor_seq_i,'Final_NHD_Hydro_Class']
    
      if dn_major_seq_i != 0:
        if (len(dn_major_class_i) > 0):
          nhd_table.loc[i,'Downstream_Hydro_Class'] = int(dn_major_class_i.iloc[0])
      elif dn_minor_seq_i != 0:
        if (len(dn_minor_class_i) > 0):
          nhd_table.loc[i,'Downstream_Hydro_Class'] = int(dn_minor_class_i.iloc[0])
  
    # assign upstream hydrologic class
    upstream_zero_rows = np.where(nhd_table['Upstream_Hydro_Class'] == 0)[0]
    for i in upstream_zero_rows:
      up_seq_i = nhd_table.loc[i,'uphydroseq'].item()
      up_class_i = nhd_table.loc[nhd_table['hydroseq'] == up_seq_i,'Final_NHD_Hydro_Class']
      if up_seq_i != 0:
        if (len(up_class_i) > 0):
          nhd_table.loc[i,'Upstream_Hydro_Class'] = int(up_class_i.iloc[0])
  
    # assign final hydrologic class
    final_zero_rows = np.where(nhd_table['Final_NHD_Hydro_Class'] == 0)[0]
    for i in final_zero_rows:
      hydro_class_i = nhd_table.loc[i,'Final_NHD_Hydro_Class'].item()
      up_class_i = nhd_table.loc[i,'Upstream_Hydro_Class'].item()
      down_class_i = nhd_table.loc[i,'Downstream_Hydro_Class'].item()
      if (hydro_class_i != 0):
        nhd_table.loc[i,'Final_NHD_Hydro_Class'] = hydro_class_i
      elif ((down_class_i != 0) & (up_class_i != 0)):
        if (down_class_i == up_class_i):
          nhd_table.loc[i,'Final_NHD_Hydro_Class'] = down_class_i
        elif (down_class_i < up_class_i):
          nhd_table.loc[i,'Final_NHD_Hydro_Class'] = up_class_i
        elif (down_class_i > up_class_i):
          nhd_table.loc[i,'Final_NHD_Hydro_Class'] = down_class_i
      elif (down_class_i != 0):
        if (down_class_i < 3):
          nhd_table.loc[i,'Final_NHD_Hydro_Class'] = down_class_i + 1
        elif (down_class_i == 3):
          nhd_table.loc[i,'Final_NHD_Hydro_Class'] = down_class_i
      elif (up_class_i != 0):
        nhd_table.loc[i,'Final_NHD_Hydro_Class'] = up_class_i   
    hydro_class_zeros.append(sum(nhd_table['Final_NHD_Hydro_Class'] == 0))

In [ ]:
# check for convergence
hydro_class_zeros

## Write results to csv

In [ ]:
nhd_table_outpath = os.path.normpath(os.path.join(nhd_gdb, "NHDFlowline_IL_Step5_HydroClass.csv"))

In [ ]:
nhd_table[['permanent_identifier','Final_NHD_Hydro_Class']].to_csv(nhd_table_outpath)

## Append results from csv to flowline shapefile

In [ ]:
# previous NHD flowline layer without filled hydrologic classifications
step4_nhd = os.path.normpath(os.path.join(nhd_gdb, "NHDFlowline_Step4_IL_WaterbodyIntersect_Classification"))

In [ ]:
# new NHD flowline layer without filled hydrologic classifications
step5_nhd = os.path.normpath(os.path.join(nhd_gdb,"NHDFlowline_Step5_IL_JoinRClassification"))

In [ ]:
arcpy.management.MakeFeatureLayer(step4_nhd, "temp_layer");
arcpy.management.AddJoin(
    in_layer_or_view="temp_layer",
    in_field="permanent_identifier",       
    join_table=nhd_table_outpath,
    join_field="permanent_identifier",
    join_type="KEEP_ALL"
);
arcpy.conversion.ExportFeatures(
    in_features="temp_layer",
    out_features=step5_nhd
);
arcpy.management.Delete("temp_layer");

## Remove flowlines without a hydrologic classification

In [ ]:
step6_nhd = os.path.normpath(os.path.join(nhd_gdb,"NHDFlowline_Step6_RemoveHydroNAs"))

In [ ]:
# copy previous layer in case of mistakes
arcpy.conversion.ExportFeatures(
    in_features = step5_nhd,
    out_features = step6_nhd,
    where_clause = "Final_NHD_Hydro_Class NOT IN (0)"
);

In [ ]:
count6 = int(arcpy.management.GetCount(step6_nhd)[0])
count6

In [ ]:
count5 = int(arcpy.management.GetCount(step5_nhd)[0])
count5

In [ ]:
count5-count6

# Join Brinkerhoff ephemeral stream data

## Add join Brinkerhoff table to most recent flowline shapefile

In [ ]:
# new file with brinkerhoff classifications
step7_nhd = os.path.normpath(os.path.join(nhd_gdb,"NHDFlowline_Step7_RemoveHydroNAs_JoinBrinkerhoff"))

In [ ]:
# convert NHDplusId field to integer
arcpy.management.CalculateField(
    in_table=step6_nhd,
    field="NHDPlusID",
    expression="int(!NHDPlusID!)",
    expression_type="PYTHON3"
);

In [ ]:
# brinkerhoff dataframe with ephemeral v. nonephemeral classifications for all flowlines
brink_df = os.path.normpath(r"F:/databases/wetlands/Flowlines, Floodplains, Levees/CONUS_ephemeral_results_v2/all_il_stream_class.csv")
for f in arcpy.ListFields(brink_df):
    print(f.name)

In [ ]:
# add join based on NHDPlusID
arcpy.management.MakeFeatureLayer(step6_nhd, "temp_layer");
arcpy.management.AddJoin(
    in_layer_or_view="temp_layer",
    in_field="nhdplusid",       
    join_table=brink_df,
    join_field="NHDPlusID",
    join_type="KEEP_ALL"
);
arcpy.conversion.ExportFeatures(
    in_features="temp_layer",
    out_features=step7_nhd
);
arcpy.management.Delete("temp_layer");

## Update NHD-based flow permanence levels with Brinkerhoff data

In [ ]:
# create new brinkerhoff NHD class column
arcpy.management.AddField(step7_nhd, "Brinkerhoff_Hydro_Class", "LONG");

In [ ]:
hydro_class_update_rule = """
def BrinkUpdate(perenniality, Final_NHD_Hydro_Class):
    if perenniality == 'ephemeral':
        return 3
    else:
        return int(Final_NHD_Hydro_Class)
"""

In [ ]:
arcpy.management.CalculateField(
    in_table = step7_nhd,
    field = "Brinkerhoff_Hydro_Class",
    expression = "BrinkUpdate(!perenniality!, !Final_NHD_Hydro_Class!)",
    expression_type = "PYTHON3",
    code_block = hydro_class_update_rule
);

# Estimate total lengths in each category for original NHD- and Brinkerhoff-informed flow permanence categories

## Original NHD

In [ ]:
# original total
step1_nhd = os.path.normpath(os.path.join(nhd_gdb, "NHDFlowline_Step1_IL_Clip_AlbersNAD1983"))
nhd_original_total = sum(row[0] for row in arcpy.da.SearchCursor(step1_nhd, ['Length_Km_Geodesic']))
round(nhd_original_total,1)

In [ ]:
nhd_total_nonisolated = sum(row[0] for row in arcpy.da.SearchCursor(step6_nhd, ['Length_Km_Geodesic']))
round(nhd_total_nonisolated,1)

In [ ]:
# non-isolated perennial
nhd_perennial = sum(row[1] for row in arcpy.da.SearchCursor(step6_nhd, ['Final_NHD_Hydro_Class','Length_Km_Geodesic']) if row[0] == 1)
round(nhd_perennial,1)
round(nhd_perennial/nhd_original_total*100,1)

In [ ]:
# non-isolated intermittent
nhd_intermittent = sum(row[1] for row in arcpy.da.SearchCursor(step6_nhd, ['Final_NHD_Hydro_Class','Length_Km_Geodesic']) if row[0] == 2)
round(nhd_intermittent,1)
round(nhd_intermittent/nhd_original_total*100,1)

In [ ]:
# non-isolated ephemeral
nhd_ephemeral = sum(row[1] for row in arcpy.da.SearchCursor(step6_nhd, ['Final_NHD_Hydro_Class','Length_Km_Geodesic']) if row[0] == 3)
round(nhd_ephemeral,1)
round(nhd_ephemeral/nhd_original_total*100,1)

In [ ]:
# isolated
nhd_isolated = nhd_original_total-nhd_total_nonisolated
round(nhd_isolated,1)

In [ ]:
round(nhd_isolated/nhd_original_total*100,1)

In [ ]:
# non-isolated intermittent or ephemeral
round(nhd_ephemeral+nhd_intermittent+nhd_isolated,1)

In [ ]:
round((nhd_ephemeral+nhd_intermittent+nhd_isolated)/nhd_original_total*100,1)

## Brinkerhoff 

In [ ]:
# non-isolated perennial
brinkerhoff_perennial = sum(row[1] for row in arcpy.da.SearchCursor(step7_nhd, ['Brinkerhoff_Hydro_Class','Length_Km_Geodesic']) if row[0] == 1)
round(brinkerhoff_perennial,1)

In [ ]:
round(brinkerhoff_perennial/nhd_original_total*100,1)

In [ ]:
# non-isolated intermittent
brinkerhoff_intermittent = sum(row[1] for row in arcpy.da.SearchCursor(step7_nhd, ['Brinkerhoff_Hydro_Class','Length_Km_Geodesic']) if row[0] == 2)
round(brinkerhoff_intermittent,1)

In [ ]:
round(brinkerhoff_intermittent/nhd_original_total*100,1)

In [ ]:
# non-isolated ephemeral
brinkerhoff_ephemeral = sum(row[1] for row in arcpy.da.SearchCursor(step7_nhd, ['Brinkerhoff_Hydro_Class','Length_Km_Geodesic']) if row[0] == 3)
round(brinkerhoff_ephemeral,1)

In [ ]:
round(brinkerhoff_ephemeral/nhd_original_total*100,1)

In [ ]:
# non-isolated intermittent or ephemeral
round(brinkerhoff_ephemeral+brinkerhoff_intermittent+nhd_isolated,1)

## Difference: Brinkerhoff - Original NHD

In [ ]:
# nonisolated perennial
round(brinkerhoff_perennial-nhd_perennial,1)

In [ ]:
round((brinkerhoff_perennial-nhd_perennial)/nhd_original_total*100,1)

In [ ]:
# non-isolated intermittent
round(brinkerhoff_intermittent-nhd_intermittent,1)

In [ ]:
round((brinkerhoff_intermittent-nhd_intermittent)/nhd_original_total*100,1)

In [ ]:
# non-isolated ephemeral
round(brinkerhoff_ephemeral-nhd_ephemeral,3)

In [ ]:
round((brinkerhoff_ephemeral-nhd_ephemeral)/nhd_original_total*100,1)

In [ ]:
# non-isolated intermittent or ephemeral
round(brinkerhoff_ephemeral+brinkerhoff_intermittent-nhd_ephemeral-nhd_intermittent,1)

In [ ]:
round((brinkerhoff_ephemeral+brinkerhoff_intermittent-nhd_ephemeral-nhd_intermittent)/nhd_original_total*100,1)

# Intersect flowlines with county shapefile to estimate total non-perennial in each category

In [18]:
# county shapefiles
il_counties_unclipped = r"C:/Users/Chels/OneDrive - University of Illinois - Urbana/Ch5_Sackett_Impacts/Wetland Mapping/TIGER_Line_Census.gdb/tl_2025_il_county_unclipped_NAD1983_Albers"
il_counties_clipped = r"C:/Users/Chels/OneDrive - University of Illinois - Urbana/Ch5_Sackett_Impacts/Wetland Mapping/TIGER_Line_Census.gdb/tl_2025_il_county_clipped_NAD1983_Albers"

In [4]:
# new file for NHD x County intersect (step 10 for consistency with ArcPro model builder steps)
step10_nhd = os.path.normpath(os.path.join(nhd_gdb,"NHDFlowline_Step10_CountyIntersect_Brinkerhoff"))

In [ ]:
# pairwise intersect
arcpy.analysis.PairwiseIntersect(
    in_features=[step7_nhd, il_counties_unclipped],
    out_feature_class=step10_nhd,
    join_attributes="ALL",
    output_type="INPUT"
);

In [ ]:
# create new fields for perennial, ephemeral, and intermittent stream lengths
arcpy.management.AddField(step10_nhd, "NHD_Perennial_Km", "DOUBLE");
arcpy.management.AddField(step10_nhd, "NHD_Intermittent_Km", "DOUBLE");
arcpy.management.AddField(step10_nhd, "NHD_Ephemeral_Km", "DOUBLE");
arcpy.management.AddField(step10_nhd, "BRF_Perennial_Km", "DOUBLE");
arcpy.management.AddField(step10_nhd, "BRF_Intermittent_Km", "DOUBLE");
arcpy.management.AddField(step10_nhd, "BRF_Ephemeral_Km", "DOUBLE");

In [ ]:
# rules for getting respective streamlength into column with corresponding flow permanence
perennial_length_rule = """
def Perennial(Length_Km_Geodesic, Hydro_Class):
    if Hydro_Class == 1:
        return Length_Km_Geodesic
    else:
        return 0.0
"""

intermittent_length_rule = """
def Intermittent(Length_Km_Geodesic, Hydro_Class):
    if Hydro_Class == 2:
        return Length_Km_Geodesic
    else:
        return 0.0
"""

ephemeral_length_rule = """
def Ephemeral(Length_Km_Geodesic, Hydro_Class):
    if Hydro_Class == 3:
        return Length_Km_Geodesic
    else:
        return 0.0
"""

In [ ]:
# NHD perennial
arcpy.management.CalculateField(
    in_table = step10_nhd,
    field = "NHD_Perennial_Km",
    expression = "Perennial(!Length_Km_Geodesic!,!Final_NHD_Hydro_Class!)",
    expression_type = "PYTHON3",
    code_block = perennial_length_rule
);

In [ ]:
# NHD intermittent
arcpy.management.CalculateField(
    in_table = step10_nhd,
    field = "NHD_Intermittent_Km",
    expression = "Intermittent(!Length_Km_Geodesic!,!Final_NHD_Hydro_Class!)",
    expression_type = "PYTHON3",
    code_block = intermittent_length_rule
);

In [ ]:
# NHD ephemeral
arcpy.management.CalculateField(
    in_table = step10_nhd,
    field = "NHD_Ephemeral_Km",
    expression = "Ephemeral(!Length_Km_Geodesic!,!Final_NHD_Hydro_Class!)",
    expression_type = "PYTHON3",
    code_block = ephemeral_length_rule
);

In [ ]:
# BRF perennial
arcpy.management.CalculateField(
    in_table = step10_nhd,
    field = "BRF_Perennial_Km",
    expression = "Perennial(!Length_Km_Geodesic!,!Brinkerhoff_Hydro_Class!)",
    expression_type = "PYTHON3",
    code_block = perennial_length_rule
);

In [ ]:
# BRF intermittent
arcpy.management.CalculateField(
    in_table = step10_nhd,
    field = "BRF_Intermittent_Km",
    expression = "Intermittent(!Length_Km_Geodesic!,!Brinkerhoff_Hydro_Class!)",
    expression_type = "PYTHON3",
    code_block = intermittent_length_rule
);

In [ ]:
# BRF ephemeral
arcpy.management.CalculateField(
    in_table = step10_nhd,
    field = "BRF_Ephemeral_Km",
    expression = "Ephemeral(!Length_Km_Geodesic!,!Brinkerhoff_Hydro_Class!)",
    expression_type = "PYTHON3",
    code_block = ephemeral_length_rule
);

In [5]:
# new file for NHD x County dissolve
step11_nhd = os.path.normpath(os.path.join(nhd_gdb,"NHDFlowline_Step11_CountyDissolve_Brinkerhoff"))

In [ ]:
# fields to add for dissolve
statistics_fields = [
    ["NHD_Perennial_Km", "SUM"], 
    ["NHD_Intermittent_Km", "SUM"],
    ["NHD_Ephemeral_Km", "SUM"],
    ["BRF_Perennial_Km", "SUM"],
    ["BRF_Intermittent_Km", "SUM"],
    ["BRF_Ephemeral_Km", "SUM"]
]

In [ ]:
# dissolve by county name and sum length in each category including total
arcpy.analysis.PairwiseDissolve(
    in_features=step10_nhd,
    out_feature_class=step11_nhd,
    dissolve_field=['NAME'],
    statistics_fields=statistics_fields,
    multi_part="MULTI_PART"
);

In [13]:
arcpy.management.DeleteField(step11_nhd, ["SUM_BRF_Nonperennial_Km"])
for f in arcpy.ListFields(step11_nhd):
    print(f.name)

OBJECTID
Shape
NAME
SUM_NHD_Perennial_Km
SUM_NHD_Intermittent_Km
SUM_NHD_Ephemeral_Km
SUM_BRF_Perennial_Km
SUM_BRF_Intermittent_Km
SUM_BRF_Ephemeral_Km
Shape_Length


In [14]:
# BRF: estimate total nonperennial length
arcpy.management.AddField(step11_nhd, "SUM_BRF_Nonperennial_Km", "DOUBLE");
arcpy.management.CalculateField(
    in_table = step11_nhd,
    field = "SUM_BRF_Nonperennial_Km",
    expression = "!SUM_BRF_Intermittent_Km! + !SUM_BRF_Ephemeral_Km!",
    expression_type = "PYTHON3"
);

In [15]:
# NHD: estimate total nonperennial length
arcpy.management.AddField(step11_nhd, "SUM_NHD_Nonperennial_Km", "DOUBLE");
arcpy.management.CalculateField(
    in_table = step11_nhd,
    field = "SUM_NHD_Nonperennial_Km",
    expression = "!SUM_NHD_Intermittent_Km! + !SUM_NHD_Ephemeral_Km!",
    expression_type = "PYTHON3"
);

In [16]:
# Brinkerhoff - NHD total nonperennial length
arcpy.management.AddField(step11_nhd, "DIFF_BRF_NHD_Nonperennial_Km", "DOUBLE");
arcpy.management.CalculateField(
    in_table = step11_nhd,
    field = "DIFF_BRF_NHD_Nonperennial_Km",
    expression = "!SUM_BRF_Nonperennial_Km! - !SUM_NHD_Nonperennial_Km!",
    expression_type = "PYTHON3"
);

In [19]:
# new final layer with stream length sums appended to counties clipped to Illinois boundary (due to lake Mi)
step12_nhd = os.path.normpath(os.path.join(nhd_gdb,"NHDFlowline_Step12_CountyJoinSums_Brinkerhoff"))

In [20]:
# join shapefile from unclipped counties to clipped counties for visualization
arcpy.management.MakeFeatureLayer(il_counties_clipped, "temp_layer");
arcpy.management.AddJoin(
    in_layer_or_view="temp_layer",
    in_field="NAME",       
    join_table=step11_nhd,
    join_field="NAME",
    join_type="KEEP_ALL"
);
arcpy.conversion.ExportFeatures(
    in_features="temp_layer",
    out_features=step12_nhd
);
arcpy.management.Delete("temp_layer");